# EpiAI 一站式教程

从原始数据到部署预测的完整流程：
1. 加载数据
2. 批量训练多种模型 + 排行榜
3. 部署 + 风险评分
4. 预测（含置信区间 + 多风险算法）
5. 新数据追加 + 选择性重训
6. 模型持久化与加载

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

from EpiAI import ForecastTask

## 1. 加载数据

In [2]:
DATA_PATH = Path.home() / "infective_disease" / "pingtai" / "merged_clean.csv"
DISEASE = "登革热 Dengue fever"

df_raw = pd.read_csv(DATA_PATH)
df_raw["Year/Month"] = pd.to_datetime(df_raw["Year/Month"])
df = (
    df_raw.query("Diseases == @DISEASE")
    [["Year/Month", "Case number"]]
    .rename(columns={"Year/Month": "time", "Case number": "cases"})
    .reset_index(drop=True)
)
df = df[df["time"].dt.year >= 2010].reset_index(drop=True)

print(f"数据: {len(df)} 行, {df['time'].min().date()} → {df['time'].max().date()}")
df.head()

数据: 196 行, 2010-01-01 → 2026-04-01


,time,cases
0,2010-01-01,3.0
1,2010-02-01,0.0
2,2010-03-01,3.0
3,2010-04-01,2.0
4,2010-05-01,4.0


## 2. 定义预测任务

`ForecastTask` 是一切的核心：指定数据、目标列、频率、预测步数。

In [3]:
task = ForecastTask(
    data=df,
    target="cases",
    time_col="time",
    freq="MS",          # 月数据
    horizon=12,          # 预测未来 12 期
)
print(f"Task: {task}")
print(f"季节周期: {task.freq_to_season_length()} 期/年")

Task: ForecastTask(target=['cases'], freq=MS, horizon=12)
季节周期: 12 期/年


## 3. 批量训练 + 排行榜

传入多个模型名，自动训练并返回 `ModelVault`（含对比指标）。

In [4]:
vault = task.compare([
    "sf_auto_ets",       # StatsForecast AutoETS
    "sf_holt_winters",   # StatsForecast Holt-Winters
    "sf_theta",          # StatsForecast Theta
    "prophet",           # Facebook Prophet
    "serfling",          # Serfling 回归（传染病基线）
    "mlp"
])

print("\n排行榜（按 R² 排序）:")
display(vault.summary())

17:33:25 - cmdstanpy - INFO - Chain [1] start processing
17:33:25 - cmdstanpy - INFO - Chain [1] done processing


  sf_auto_ets                     MAE=1236  R²=-0.076  PearsonR=nan
  sf_holt_winters                 MAE=733  R²=0.470  PearsonR=0.849
  sf_theta                        MAE=1713  R²=-0.037  PearsonR=-0.062
  prophet                         MAE=715  R²=0.441  PearsonR=0.841
  serfling                        MAE=741  R²=0.382  PearsonR=0.774
  mlp                             MAE=842  R²=0.257  PearsonR=0.753

排行榜（按 R² 排序）:


,paradigm,MAE,RMSE,MAPE,R2,PearsonR,n
model,,,,,,,
sf_holt_winters,ts,733.159607,1675.256470,221.675920,0.470116,0.848855,30
prophet,ts,714.834167,1720.506958,174.477649,0.441104,0.841314,30
serfling,ts,740.600586,1809.879761,195.742154,0.381532,0.773599,30
mlp,torch,841.799744,1983.913452,144.377542,0.256872,0.753197,30
sf_theta,ts,1713.466675,2344.056885,1442.966938,-0.037419,-0.062186,30
sf_auto_ets,ts,1236.257324,2386.957031,484.036303,-0.075740,NaN,30


In [11]:
vault.models['sf_auto_ets'].forecast(steps=30)

array([[[1131.6116 ]],

       [[2136.5195 ]],

       [[4198.952  ]],

       [[1235.9838 ]],

       [[ 437.71094]],

       [[ 328.9765 ]],

       [[ 319.09842]],

       [[ 295.40298]],

       [[ 358.71786]],

       [[ 423.26266]],

       [[ 409.5273 ]],

       [[ 618.8732 ]],

       [[1131.6116 ]],

       [[2136.5195 ]],

       [[4198.952  ]],

       [[1235.9838 ]],

       [[ 437.71094]],

       [[ 328.9765 ]],

       [[ 319.09842]],

       [[ 295.40298]],

       [[ 358.71786]],

       [[ 423.26266]],

       [[ 409.5273 ]],

       [[ 618.8732 ]],

       [[1131.6116 ]],

       [[2136.5195 ]],

       [[4198.952  ]],

       [[1235.9838 ]],

       [[ 437.71094]],

       [[ 328.9765 ]]], dtype=float32)

## 4. 部署 + 风险评分

选择最佳模型部署。可以同时启用多个风险算法。

In [5]:
best_model = vault.best("R2")
print(f"最佳模型: {best_model}")

runtime = task.deploy(
    best_model,
    vault,
    risk=["quantile", "zscore"],  # 同时用分位数和 Z-score 评分
)
print(f"Runtime: {runtime}")
print(f"风险算法: {list(runtime.risk_scorers.keys())}")

最佳模型: sf_holt_winters
Runtime: DeploymentRuntime(6 models, 196 rows)
风险算法: ['quantile', 'zscore']


## 5. 预测

支持置信区间（`level` 参数）和多种风险评分。

In [6]:
report = runtime.predict(horizon=6, level=95)
print("输出列:", report.columns.tolist())
display(report)

输出列: ['time', 'sf_auto_ets', 'sf_holt_winters', 'sf_theta', 'prophet', 'serfling', 'mlp', 'sf_auto_ets_lo', 'sf_auto_ets_hi', 'sf_holt_winters_lo', 'sf_holt_winters_hi', 'sf_theta_lo', 'sf_theta_hi', 'sf_theta_quantile', 'sf_holt_winters_quantile', 'serfling_quantile', 'mlp_quantile', 'sf_auto_ets_quantile', 'prophet_quantile', 'sf_theta_zscore', 'sf_holt_winters_zscore', 'serfling_zscore', 'mlp_zscore', 'sf_auto_ets_zscore', 'prophet_zscore']


,time,sf_auto_ets,sf_holt_winters,sf_theta,prophet,serfling,mlp,sf_auto_ets_lo,sf_auto_ets_hi,sf_holt_winters_lo,...,serfling_quantile,mlp_quantile,sf_auto_ets_quantile,prophet_quantile,sf_theta_zscore,sf_holt_winters_zscore,serfling_zscore,mlp_zscore,sf_auto_ets_zscore,prophet_zscore
0,2026-05-01,586.372192,523.064819,1686.264893,303.264435,304.151764,-39.195126,-4320.043945,5492.788086,-4201.188965,...,3,0,3,3,3,3,3,0,3,3
1,2026-06-01,586.372192,376.585632,1687.964966,333.766785,106.748489,36.531624,-4320.043945,5492.788086,-4347.707031,...,1,0,3,2,3,1,0,0,2,1
2,2026-07-01,586.372192,517.575989,1689.665039,517.258789,494.774658,237.359726,-4320.043945,5492.788086,-4206.757812,...,1,1,1,1,2,0,0,0,0,0
3,2026-08-01,586.372192,1020.706299,1691.365112,1050.610107,1452.620728,288.533600,-4320.043945,5492.788574,-3703.670898,...,1,0,0,1,0,0,0,0,0,0
4,2026-09-01,586.372192,1868.620117,1693.065186,1882.290161,2349.788330,485.303009,-4320.043945,5492.788574,-2855.802979,...,1,0,0,1,0,0,0,0,0,0
5,2026-10-01,586.372192,3738.634521,1694.765259,3721.049316,2484.385010,456.045105,-4320.043945,5492.788574,-985.837036,...,0,0,0,1,0,0,0,0,0,0


### 说明

| 列 | 含义 |
|----|------|
| `time` | 预测时间点 |
| `{model}` | 预测值（均值） |
| `{model}_lo` | 95% 置信区间下限 |
| `{model}_hi` | 95% 置信区间上限 |
| `{model}_quantile` | 分位数法风险等级 0-3 |
| `{model}_zscore` | Z-score 法风险等级 0-3 |

风险等级：0=低 🟢 1=中 🟡 2=高 🟠 3=极高 🔴

## 6. 新数据接收 + 选择性重训

模拟新数据到达：先 `feed` 追加，再 `refit` 重训。

In [7]:
# 模拟新数据
np.random.seed(42)
new_data = pd.DataFrame({
    "time": pd.date_range("2026-05-01", periods=3, freq="MS"),
    "cases": np.random.randn(3) * 300 + 500,
})

runtime.feed(new_data)
print(f"data_table 现有 {len(runtime.data_table)} 行")

# 选择性重训（只重训 AutoETS，Holt-Winters 保持不变）
runtime.refit(model_names=["sf_auto_ets"], verbose=True)

print("\n重训后预测:")
display(runtime.predict(horizon=24))

data_table 现有 199 行
  refit sf_auto_ets ... OK
  risk_quantile refit OK
  risk_zscore refit OK

全部重训完成，共 6 个模型。

重训后预测:


,time,sf_auto_ets,sf_holt_winters,sf_theta,prophet,serfling,mlp,sf_theta_quantile,sf_holt_winters_quantile,serfling_quantile,mlp_quantile,sf_auto_ets_quantile,prophet_quantile,sf_theta_zscore,sf_holt_winters_zscore,serfling_zscore,mlp_zscore,sf_auto_ets_zscore,prophet_zscore
0,2026-08-01,1131.611572,1020.706299,1691.365112,1050.610107,1452.620728,510.598907,1,1,1,0,1,1,0,0,0,0,0,0
1,2026-09-01,2136.519531,1868.620117,1693.065186,1882.290161,2349.788330,703.674866,0,1,1,0,1,1,0,0,0,0,0,0
2,2026-10-01,4198.952148,3738.634521,1694.765259,3721.049316,2484.385010,807.321594,0,1,0,0,1,1,0,0,0,0,0,0
3,2026-11-01,1235.983765,1085.674927,1696.465332,842.047607,1733.401489,407.271942,1,1,1,0,1,0,0,0,0,0,0,0
4,2026-12-01,437.710938,342.147583,1698.165405,311.758667,673.324951,234.010056,3,3,3,2,3,3,3,1,3,1,2,1
5,2027-01-01,328.976501,372.130402,1699.865356,300.605560,51.116680,-115.633110,3,3,1,0,3,3,3,3,0,0,3,3
6,2027-02-01,319.098419,413.292450,1701.565430,304.836975,121.858696,32.202419,3,3,3,1,3,3,3,3,3,0,3,3
7,2027-03-01,295.402985,254.652954,1703.265503,346.841705,492.751495,-69.374931,3,3,3,0,3,3,3,3,3,0,3,3
8,2027-04-01,358.717865,330.224365,1704.965576,339.846893,602.917480,20.960745,3,3,3,0,3,3,3,3,3,0,3,3
9,2027-05-01,423.262665,547.990723,1706.665649,311.754639,335.893555,93.306084,3,3,2,1,3,2,3,2,1,0,2,1


## 7. 持久化与加载

保存到磁盘 → 服务器加载 → 恢复预测。

In [ ]:
# 保存
save_path = "/tmp/epiai_tutorial_runtime"
runtime.save(save_path)
print(f"已保存至 {save_path}")
print(f"目录内容: {list(Path(save_path).iterdir())}")

In [ ]:
# 服务器加载
from EpiAI.inference import DeploymentRuntime

loaded = DeploymentRuntime.load(save_path)
print(f"加载完成: {loaded}")
print(f"风险算法: {list(loaded.risk_scorers.keys())}")

# 恢复预测
report = loaded.predict(horizon=6, level=95)
display(report)

## 总结

```python
from EpiAI import ForecastTask

# 训练 + 对比
task = ForecastTask(data=df, target="cases", freq="MS", horizon=12)
vault = task.compare(["sf_auto_ets", "sf_holt_winters", "prophet"])

# 部署
runtime = task.deploy(vault.best("R2"), vault, risk=["quantile", "zscore"])

# 生产
report = runtime.predict(horizon=6, level=95)
runtime.feed(new_data)
runtime.refit(model_names=["sf_auto_ets"])
runtime.save("/data/runtime/")
```